In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()
import joblib, shap
from training.feature_engineering import load_xy

# 07 - SHAP explainability (XGBoost delay classifier)
`TreeExplainer` on the raw margin of the production model; the API returns the top 5 factors per prediction.

In [2]:
model = joblib.load(C.MODELS_DIR / "xgb_delay_clf.joblib")
bg = pd.read_parquet(C.MODELS_DIR / "shap_background.parquet")
X, _ = load_xy("test", "delay", sample=0.003)
expl = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
sv = expl.shap_values(X.iloc[:500])
imp = pd.Series(np.abs(sv).mean(0), index=X.columns).sort_values()[-15:]
fig, ax = plt.subplots(figsize=(7, 5)); imp.plot.barh(ax=ax, color="#7c3aed"); ax.set_title("mean |SHAP| (test sample)"); save("shap_global_importance")

In [3]:
i = 0
row = pd.Series(sv[i], index=X.columns); top = row.reindex(row.abs().sort_values(ascending=False).index)[:5]
pd.DataFrame({"feature": top.index, "value": X.iloc[i][top.index].values, "shap": top.values.round(3),
              "direction": np.where(top.values > 0, "increases_delay_risk", "decreases_delay_risk")})

,feature,value,shap,direction
0,airline_delay_roll,0.375690,0.499,increases_delay_risk
1,origin_time_delay_rate,0.130987,-0.384,decreases_delay_risk
2,airline_cancel_roll,0.035284,-0.165,decreases_delay_risk
3,dep_time_idx,1.000000,-0.132,decreases_delay_risk
4,dest_delay_roll,0.325614,0.115,increases_delay_risk


In [4]:
shap.summary_plot(sv, X.iloc[:500], show=False, max_display=15); plt.tight_layout(); plt.savefig(IMG / "shap_summary.png", dpi=110); plt.show()